# CacheBackedEmbeddings (2026 업데이트판)

임베딩 결과를 키-값 저장소에 캐싱해, 같은 텍스트를 다시 임베딩할 때 API 호출을 생략합니다. 텍스트를 해시한 값이 캐시 키가 됩니다.

### 원본 대비 변경 사항
| 항목 | 원본 | 현재 권장 |
|---|---|---|
| `CacheBackedEmbeddings` | `langchain.embeddings` | `langchain_classic.embeddings` (LangChain v1에서 이동) |
| `LocalFileStore` | `langchain.storage` | `langchain_classic.storage` |
| `InMemoryByteStore` | `langchain.storage` | `langchain_core.stores` |
| 캐시 키 해시 | 기본값 SHA-1 (경고 발생) | `key_encoder="sha256"` 명시 |
| 쿼리 캐싱 | 없음 | `query_embedding_cache=True` 로 쿼리도 캐싱 |
| 문서 로더 | `langchain.document_loaders.TextLoader` | 파이썬 기본 파일 읽기 + `text_splitter.create_documents()` |
| 벡터 저장소 | `langchain_community` 의 `FAISS` | `langchain_core` 의 `InMemoryVectorStore` |

**`langchain-community`에 대해**: 이 패키지는 2026년 5월 공식적으로 지원 종료(sunset)되었고 저장소도 보관(archive) 처리되었습니다. 설치는 여전히 되지만 더 이상 수정되지 않으므로, 새 코드에서는 `langchain-core` 또는 공급자별 전용 패키지(`langchain-openai`, `langchain-chroma` 등)를 사용합니다.

`from_bytes_store`의 주요 매개변수:
- `underlying_embeddings`: 실제 임베딩을 계산할 모델
- `document_embedding_cache`: 문서 임베딩을 저장할 `ByteStore`
- `namespace`: 캐시 충돌 방지용 이름 공간 — **임베딩 모델 이름으로 반드시 설정**하세요. 같은 텍스트라도 모델이 다르면 벡터가 다르기 때문입니다.
- `query_embedding_cache`: `True`면 쿼리 임베딩도 같은 저장소에 캐싱
- `key_encoder`: 키 해시 방식 (`"sha256"`, `"blake2b"`, `"sha512"` 또는 함수)

In [ ]:
%pip install -qU langchain-classic langchain-core langchain-openai langchain-text-splitters python-dotenv

## 환경 설정

- `.env` 파일의 API 키를 `python-dotenv`로 불러옵니다.
- **변경점**: 책에서 사용한 `langchain_teddynote.logging.langsmith()`는 서드파티 헬퍼입니다. 현재 LangSmith 공식 방식은 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)만 설정하는 것이며, 별도 패키지가 필요 없습니다.
- 참고: 임베딩 호출(`embed_query`, `embed_documents`)은 Runnable이 아니어서 LangSmith에 트레이스가 남지 않습니다. 이 챕터에서는 없어도 되는 설정이지만, 이후 체인/에이전트 실습과 형태를 맞추기 위해 둡니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 키를 환경 변수로 로드

# LangSmith 추적 (LANGSMITH_API_KEY 는 .env 에 넣어 둡니다)
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "CH08-Embeddings")

## LocalFileStore 사용 (영구 보관)

로컬 파일 시스템에 임베딩을 저장합니다. 노트북을 재시작해도 캐시가 유지됩니다.

In [ ]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings

# 실제 임베딩 모델 (모델명을 명시해 두면 namespace로 쓰기 좋습니다)
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# 로컬 파일 저장소
store = LocalFileStore("./cache/")

# 캐시 지원 임베딩
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding,
    document_embedding_cache=store,
    namespace=embedding.model,     # 모델별로 캐시 공간 분리
    query_embedding_cache=True,    # 쿼리 임베딩도 캐싱 (기본값 False)
    key_encoder="sha256",          # 기본값 SHA-1 대신 충돌에 강한 해시 사용
)

In [ ]:
# 캐시가 비어 있는지 확인
list(store.yield_keys())[:5]

문서를 로드하고 청크로 분할합니다.

**변경점**:
- `TextLoader`(`langchain_community`)는 단순히 파일을 읽어 `Document`로 감싸는 역할이므로, 파이썬 기본 기능으로 읽고 `create_documents()`로 바로 분할합니다. 출처 정보는 `metadatas`로 넣습니다.
- `CharacterTextSplitter`는 구분자 하나(`\n\n`)로만 자르므로 청크 크기를 넘는 경우가 생깁니다. 일반 텍스트에는 여러 구분자를 순서대로 시도하는 `RecursiveCharacterTextSplitter`가 기본 권장입니다.

In [ ]:
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path = Path("./data/appendix-keywords.txt")
raw_text = file_path.read_text(encoding="utf-8")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.create_documents(
    [raw_text], metadatas=[{"source": str(file_path)}]
)
len(documents)

벡터 저장소를 생성합니다. 첫 실행에서는 모든 청크를 실제로 임베딩(API 호출)합니다.

**변경점**: `FAISS`(`langchain_community`) 대신 `langchain_core`에 내장된 `InMemoryVectorStore`를 사용합니다. 추가 설치가 필요 없고 학습용으로 충분합니다. 운영 환경에서는 `langchain-chroma`, `langchain-qdrant`, `langchain-postgres` 등 전용 패키지를 사용하세요.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

%time db = InMemoryVectorStore.from_documents(documents, cached_embedder)

같은 문서로 벡터 저장소를 다시 만들면, 임베딩을 캐시에서 읽어 오므로 훨씬 빠릅니다.

In [ ]:
%time db2 = InMemoryVectorStore.from_documents(documents, cached_embedder)

In [ ]:
# 캐시에 저장된 키 확인 (namespace + sha256 해시 형태)
keys = list(store.yield_keys())
print(len(keys))
keys[:3]

### 쿼리 캐싱 확인

`query_embedding_cache=True`로 설정했으므로 같은 질문을 두 번째로 검색할 때는 쿼리 임베딩 API 호출이 생략됩니다.

In [ ]:
query = "임베딩이란 무엇인가요?"

%time _ = db.similarity_search(query, k=2)   # 첫 호출: 쿼리 임베딩 계산
%time results = db.similarity_search(query, k=2)  # 두 번째 호출: 캐시 사용

for doc in results:
    print(doc.page_content[:200], "\n---")

## `InMemoryByteStore` 사용 (비영구적)

다른 `ByteStore`를 쓰려면 `from_bytes_store`에 해당 저장소만 넘기면 됩니다. 메모리 저장소는 프로세스가 끝나면 사라지므로 테스트용으로 적합합니다.

**변경점**: `InMemoryByteStore`는 이제 `langchain_core.stores`에서 가져옵니다.

In [ ]:
from langchain_core.stores import InMemoryByteStore

mem_store = InMemoryByteStore()

cached_embedder_mem = CacheBackedEmbeddings.from_bytes_store(
    embedding,
    mem_store,
    namespace=embedding.model,
    query_embedding_cache=True,
    key_encoder="sha256",
)

_ = cached_embedder_mem.embed_documents(["안녕하세요", "반갑습니다"])
list(mem_store.yield_keys())